# Heat on a square plate, by Jacobi iteration

A thin square plate is described by an `n` by `n` grid of temperatures. The temperatures on the
outer edge of the grid are held fixed. In each sweep of the Jacobi method, every interior
temperature is replaced by the average of its four neighbours, and only the temperatures from
the previous sweep are used. Repeated sweeps carry the heat in from the edges and smooth the
plate towards its steady state. The function below applies a given number of sweeps and returns
the new grid. The demonstration applies 200 sweeps to a plate with a hot top edge and three cold
edges, and prints the temperatures of the first six points down the middle of the plate.

In [ ]:
import numpy as np


def jacobi(u, iters):
    """Apply `iters` Jacobi sweeps to the grid `u`, whose outer edge stays fixed."""
    u = u.copy()
    for _ in range(iters):
        u[1:-1, 1:-1] = 0.25 * (u[:-2, 1:-1] + u[2:, 1:-1] + u[1:-1, :-2] + u[1:-1, 2:])
    return u

In [ ]:
print(jacobi(np.vstack([np.ones((1, 64)), np.zeros((63, 64))]), 200)[:6, 32])

The declaration below tells the harness how to call the function and how to check a routine
against it. The grid `u` is both an argument and a result, so a routine updates it in place. The
C routine takes the grid, then its side `n`, then the number of sweeps. `make_inputs` draws a
different plate for every call: random temperatures, a few of them exactly 0 or 1, with each
edge held at a single temperature, which is sometimes exactly 0 or 1 as well. The number of
sweeps is between 300 and 600. The tolerance is 1e-14 of the largest temperature. A routine that
adds the four neighbours in a different order differs from the function by about 7e-16.

In [ ]:
def make_inputs(size, rng):
    u = rng.uniform(0.0, 1.0, (size, size))
    extreme = rng.random((size, size)) < 0.05
    u[extreme] = rng.integers(0, 2, extreme.sum())
    edges = np.where(rng.random(4) < 0.4, rng.integers(0, 2, 4), rng.uniform(0.0, 1.0, 4))
    u[0, :], u[-1, :], u[:, 0], u[:, -1] = edges
    return {"u": u, "iters": int(rng.integers(300, 601))}


ROUTINE = {
    "function": jacobi,
    "make_inputs": make_inputs,
    "arguments": {"u": "float64[n, n]", "iters": "int32"},
    "results": {"u": "float64[n, n]"},
    "c_order": ["u", "n", "iters"],
    "test_sizes": [1, 8, 64, 256],
    "benchmark_size": 1024,
    "tolerance": 1e-14,
}